In [1]:
import pandas as pd
import os

In [2]:
expr = pd.read_csv('../data/raw/HiSeqV2', sep='\t', index_col=0)
print("Expression shape:", expr.shape)
print(expr.iloc[:5, :5])

Expression shape: (20530, 1218)
           TCGA-AR-A5QQ-01  TCGA-D8-A1JA-01  TCGA-BH-A0BQ-01  TCGA-BH-A0BT-01  \
sample                                                                          
ARHGEF10L           9.5074           7.4346           9.3216           9.0198   
HIF3A               1.5787           3.6607           2.7224           1.3414   
RNF17               0.0000           0.6245           0.5526           0.0000   
RNF10              11.3676          11.9181          11.9665          13.1881   
RNF11              11.1292          13.5273          11.4105          11.0911   

           TCGA-A8-A06X-01  
sample                      
ARHGEF10L           9.6417  
HIF3A               0.5819  
RNF17               0.0000  
RNF10              12.0036  
RNF11              11.2545  


In [3]:
mut = pd.read_csv('../data/raw/BRCA_mc3_gene_level.txt', sep='\t', index_col=0)
print("Mutation shape:", mut.shape)
print(mut.iloc[:5, :5])

Mutation shape: (40543, 791)
         TCGA-3C-AAAU-01  TCGA-3C-AALI-01  TCGA-3C-AALJ-01  TCGA-3C-AALK-01  \
sample                                                                        
UBE2Q2                 0                0                0                0   
CHMP1B                 0                0                0                0   
PSMA2P1                0                0                0                0   
SHQ1P1                 0                0                0                0   
CPHL1P                 0                0                0                0   

         TCGA-4H-AAAK-01  
sample                    
UBE2Q2                 0  
CHMP1B                 0  
PSMA2P1                0  
SHQ1P1                 0  
CPHL1P                 0  


In [4]:
clin = pd.read_csv('../data/raw/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix', sep='\t', index_col=0)
print("Clinical shape:", clin.shape)
print(clin.columns.tolist())

Clinical shape: (1247, 193)
['AJCC_Stage_nature2012', 'Age_at_Initial_Pathologic_Diagnosis_nature2012', 'CN_Clusters_nature2012', 'Converted_Stage_nature2012', 'Days_to_Date_of_Last_Contact_nature2012', 'Days_to_date_of_Death_nature2012', 'ER_Status_nature2012', 'Gender_nature2012', 'HER2_Final_Status_nature2012', 'Integrated_Clusters_no_exp__nature2012', 'Integrated_Clusters_unsup_exp__nature2012', 'Integrated_Clusters_with_PAM50__nature2012', 'Metastasis_Coded_nature2012', 'Metastasis_nature2012', 'Node_Coded_nature2012', 'Node_nature2012', 'OS_Time_nature2012', 'OS_event_nature2012', 'PAM50Call_RNAseq', 'PAM50_mRNA_nature2012', 'PR_Status_nature2012', 'RPPA_Clusters_nature2012', 'SigClust_Intrinsic_mRNA_nature2012', 'SigClust_Unsupervised_mRNA_nature2012', 'Survival_Data_Form_nature2012', 'Tumor_T1_Coded_nature2012', 'Tumor_nature2012', 'Vital_Status_nature2012', '_INTEGRATION', '_PANCAN_CNA_PANCAN_K8', '_PANCAN_Cluster_Cluster_PANCAN', '_PANCAN_DNAMethyl_BRCA', '_PANCAN_DNAMethyl_P

In [5]:
# Check the PAM50 label column
print(clin['PAM50Call_RNAseq'].value_counts())
print("Non-null PAM50 labels:", clin['PAM50Call_RNAseq'].notna().sum())

# Check sample ID overlap between the three datasets
expr_samples = set(expr.columns)
mut_samples = set(mut.columns)
clin_samples = set(clin.index)

print("Expression samples:", len(expr_samples))
print("Mutation samples:", len(mut_samples))
print("Clinical samples:", len(clin_samples))

# Patients present in ALL THREE with a non-null PAM50 label
labeled_patients = set(clin[clin['PAM50Call_RNAseq'].notna()].index)
common = expr_samples & mut_samples & labeled_patients
print("Patients in all 3 datasets with PAM50 label:", len(common))

PAM50Call_RNAseq
LumA      434
LumB      194
Basal     142
Normal    119
Her2       67
Name: count, dtype: int64
Non-null PAM50 labels: 956
Expression samples: 1218
Mutation samples: 791
Clinical samples: 1247
Patients in all 3 datasets with PAM50 label: 593


In [6]:
# Convert to sorted list for consistent ordering
common_patients = sorted(common)

# Subset and transpose expression + mutation so patients are rows
expr_subset = expr[common_patients].T   # now rows=patients, cols=genes
mut_subset = mut[common_patients].T

# Subset clinical to get labels, same patient order
labels = clin.loc[common_patients, 'PAM50Call_RNAseq']

print("Expression subset:", expr_subset.shape)
print("Mutation subset:", mut_subset.shape)
print("Labels:", labels.shape)
print(labels.value_counts())

# Sanity check: indices match across all three
print("Index match (expr vs labels):", (expr_subset.index == labels.index).all())
print("Index match (mut vs labels):", (mut_subset.index == labels.index).all())

Expression subset: (593, 20530)
Mutation subset: (593, 40543)
Labels: (593,)
PAM50Call_RNAseq
LumA      301
LumB      125
Basal     106
Her2       42
Normal     19
Name: count, dtype: int64
Index match (expr vs labels): True
Index match (mut vs labels): True
